# Phase 4.2 — Cold Start & Error Analysis

Define practical strategies for new users and new products, inspect sparse-user and sparse-product behavior, and record the main model limitations and findings.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
TRAIN_UI_PATH = PROCESSED_DIR / "train_user_item.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

print("Project root:", PROJECT_ROOT)


Project root: f:\annuspeaks.com\recommendation-system


## 4.2.1 Load Interaction Data


In [2]:
train = pd.read_csv(
    TRAIN_PATH,
    usecols=["user_id", "item_id", "weight", "timestamp"],
)

train_ui = pd.read_csv(
    TRAIN_UI_PATH,
    usecols=["user_id", "item_id", "total_weight"],
)

test = pd.read_csv(
    TEST_PATH,
    usecols=["user_id", "item_id"],
)

user_activity = (
    train_ui.groupby("user_id")
    .size()
    .rename("interaction_count")
)

item_activity = (
    train_ui.groupby("item_id")
    .size()
    .rename("user_count")
)

popular_items = (
    train_ui.groupby("item_id")["total_weight"]
    .sum()
    .sort_values(ascending=False)
    .index
    .to_numpy()
)

print("Users:", f"{train_ui['user_id'].nunique():,}")
print("Products:", f"{train_ui['item_id'].nunique():,}")
print("Interactions:", f"{len(train_ui):,}")


Users: 1,407,580
Products: 228,392
Interactions: 1,939,777


## 4.2.2 New-User Recommendation Strategy

For a user with no interaction history, use a popularity-based fallback.

If onboarding preferences are available later, those preferences can be used to narrow the initial catalog before personalization begins.


In [3]:
def new_user_recommend(k=10):
    # Cold-start fallback: globally popular products.
    return popular_items[:k].tolist()

new_user_example = new_user_recommend(10)

print("New-user Top-K:")
print(new_user_example)


New-user Top-K:
[461686, 187946, 5411, 370653, 219512, 257040, 320130, 7943, 96924, 298009]


## 4.2.3 New-Product Recommendation Strategy

A new product can enter recommendation through its available metadata/content representation even when it has no historical interactions.

The content-based pipeline from Phase 3.2 provides the intended strategy.


In [4]:
# Identify metadata-only products using the Phase 3.2 catalog artifacts
# when available. This notebook does not rebuild the large TF-IDF index.

RAW_DIR = PROJECT_ROOT / "data" / "raw"
metadata_files = [
    RAW_DIR / "item_properties_part1.csv",
    RAW_DIR / "item_properties_part2.csv",
]

metadata_item_ids = set()

for path in metadata_files:
    if path.exists():
        for chunk in pd.read_csv(
            path,
            usecols=["itemid"],
            chunksize=250_000,
        ):
            metadata_item_ids.update(chunk["itemid"].dropna().unique())

train_item_ids = set(train_ui["item_id"].unique())

metadata_only_count = len(metadata_item_ids - train_item_ids)

print(
    "Metadata-only products:",
    f"{metadata_only_count:,}"
)
print(
    "New-product strategy:",
    "content-based metadata recommendation"
)


Metadata-only products: 237,159
New-product strategy: content-based metadata recommendation


## 4.2.4 Sparse-User and Sparse-Product Analysis

Define sparse users as users with very few observed training interactions and sparse products as products with very few observed users.

Use simple thresholds that are easy to interpret and reproduce.


In [5]:
SPARSE_USER_THRESHOLD = 2
SPARSE_PRODUCT_THRESHOLD = 2

sparse_users = user_activity[
    user_activity <= SPARSE_USER_THRESHOLD
]

sparse_products = item_activity[
    item_activity <= SPARSE_PRODUCT_THRESHOLD
]

user_count = len(user_activity)
item_count = len(item_activity)

print(
    "Sparse users:",
    f"{len(sparse_users):,}",
    f"({len(sparse_users) / user_count:.2%})"
)

print(
    "Sparse products:",
    f"{len(sparse_products):,}",
    f"({len(sparse_products) / item_count:.2%})"
)


Sparse users: 1,351,614 (96.02%)
Sparse products: 116,207 (50.88%)


In [6]:
# Simple sparse-segment diagnostics.

sparse_user_rows = train_ui[
    train_ui["user_id"].isin(sparse_users.index)
]

sparse_product_rows = train_ui[
    train_ui["item_id"].isin(sparse_products.index)
]

diagnostics = pd.DataFrame([
    {
        "segment": "Sparse Users",
        "entities": len(sparse_users),
        "interaction_rows": len(sparse_user_rows),
        "recommended_handling":
            "Use popularity/content fallback until enough history exists.",
    },
    {
        "segment": "Sparse Products",
        "entities": len(sparse_products),
        "interaction_rows": len(sparse_product_rows),
        "recommended_handling":
            "Use content signals and avoid relying only on collaborative history.",
    },
])

display(diagnostics)


,segment,entities,interaction_rows,recommended_handling
0,Sparse Users,1351614,1513520,Use popularity/content fallback until enough h...
1,Sparse Products,116207,151887,Use content signals and avoid relying only on ...


## 4.2.5 Model Limitations and Final Findings

Record the practical limitations observed during development:

- Retailrocket interactions are implicit behavioral signals rather than explicit ratings.
- The dataset is highly sparse and long-tailed.
- Popularity is a useful fallback but can over-expose head products.
- Collaborative filtering needs interaction history and therefore struggles with unseen users/products.
- Content-based recommendation depends on usable product metadata.
- The current hybrid weights are practical development weights, not learned ranking weights.
- Offline metrics on a deterministic sample are useful for comparison but are not a substitute for online A/B testing.


In [7]:
findings = {
    "feedback_type": "Implicit behavioral interactions",
    "cold_start_user": "Popularity/trending fallback; onboarding preferences can be added later",
    "cold_start_product": "Metadata/content-based recommendation",
    "sparse_user_handling": "Fallback + personalization after sufficient history",
    "sparse_product_handling": "Content signals + controlled exposure",
    "hybrid_weights": "Fixed development weights; not learned",
    "evaluation_limit": "Offline deterministic evaluation; online testing still required",
}

findings_df = pd.DataFrame(
    findings.items(),
    columns=["finding", "decision"]
)

display(findings_df)


,finding,decision
0,feedback_type,Implicit behavioral interactions
1,cold_start_user,Popularity/trending fallback; onboarding prefe...
2,cold_start_product,Metadata/content-based recommendation
3,sparse_user_handling,Fallback + personalization after sufficient hi...
4,sparse_product_handling,Content signals + controlled exposure
5,hybrid_weights,Fixed development weights; not learned
6,evaluation_limit,Offline deterministic evaluation; online testi...


In [8]:
# Final validation

assert len(new_user_example) == 10
assert metadata_only_count >= 0
assert len(sparse_users) <= user_count
assert len(sparse_products) <= item_count
assert len(findings) >= 6

print("Phase 4.2 validation: PASS")
print("New-user fallback: popularity")
print("New-product strategy: content/metadata")
print("Sparse-user threshold:", SPARSE_USER_THRESHOLD)
print("Sparse-product threshold:", SPARSE_PRODUCT_THRESHOLD)


Phase 4.2 validation: PASS
New-user fallback: popularity
New-product strategy: content/metadata
Sparse-user threshold: 2
Sparse-product threshold: 2


## Phase 4.2 Completion

- New-user recommendation strategy defined.
- New-product recommendation strategy defined.
- Sparse-user and sparse-product behavior analyzed.
- Model limitations and final findings documented.
